# Cyclic Peptide Permeability: MLP Baseline

A small fully connected network trained on ECFP4 fingerprints, as a deep-learning counterpart to the classical models in `01_classical_ml.ipynb`. Batch size, number of epochs, learning rate and hidden width were tuned with a Weights & Biases Bayesian sweep (`SWEEP_SEARCH`), and the final cell re-runs the best configuration it found (`SWEEP_BEST`).

**Result.** One hidden layer of 4 units, trained for 50 epochs with Adam (learning rate ≈ 2.1 × 10⁻³), reaches **R² = 0.313** (MAE 0.607) on a 20% hold-out split. This is in line with LightGBM on the same fingerprints (hold-out R² 0.296 on a different random split).

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

DATA_DIR = Path("../data")
SEED = 67
PROJECT_NAME = "gnn-cycpeptmpdb-v6"  # W&B project

### Sweep configurations

`SWEEP_SEARCH` is the Bayesian search over the MLP's hyperparameters, and `SWEEP_BEST` re-runs the best configuration found. Both minimise the test-set MSE logged by `evaluate`.

In [ ]:
SWEEP_SEARCH = {
    "method": "bayes",
    "name": "2-layer-MLP+ECFP+properties",
    "metric": {"goal": "minimize", "name": "MSE"},
    "run_cap": 100,
    "parameters": {
        "batch_size": {"values": [32, 64]},
        "epochs": {"values": [50, 100, 150]},
        "learning_rate": {"min": 1e-5, "max": 1e-1},
        "hidden_size": {"values": [4, 16, 32, 64, 128]},
    },
}

SWEEP_BEST = {
    "method": "grid",
    "name": "run-2-layer-MLP+ECFP",
    "metric": {"goal": "minimize", "name": "MSE"},
    "run_cap": 1,
    "parameters": {
        "batch_size": {"value": 64},
        "epochs": {"value": 50},
        "learning_rate": {"value": 0.0021464013511416644},
        "hidden_size": {"value": 4},
    },
}

## Data

ECFP4 fingerprints (cached in `data/` after the first run) and, optionally, the 210 RDKit descriptors used in `01_classical_ml.ipynb`.

In [ ]:
def generate_mfp(smiles, radius=2, n_bits=2048):
    """2,048-bit ECFP4 (Morgan, radius 2) fingerprint; all zeros if RDKit cannot parse the SMILES."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(n_bits)
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    return np.array(gen.GetFingerprint(mol))


cycdf = pd.read_csv(DATA_DIR / "CycPeptMPDB_Peptide_All.csv", low_memory=False)

mfp_path = DATA_DIR / "mfp_feature.npy"
if mfp_path.is_file():
    cycdf["Morgan_FP"] = np.load(mfp_path, allow_pickle=True)
else:
    cycdf["Morgan_FP"] = cycdf["SMILES"].apply(generate_mfp)
    np.save(mfp_path, cycdf["Morgan_FP"].to_numpy())

In [ ]:
X_ecfp = np.stack(cycdf["Morgan_FP"].values)
# ECFP4 + 210 standardised descriptors (columns MaxEStateIndex ... PC2)
X_ecfp_desc = np.hstack([X_ecfp, StandardScaler().fit_transform(cycdf.iloc[:, 35:245])])
y = cycdf["Permeability"].values

X = X_ecfp  # input used by the sweep below

In [42]:
X_ecfp_desc.shape

(8466, 2258)

## Model

In [ ]:
class PermeabilityMLP(nn.Module):
    """Perceptron with one hidden layer: input -> hidden (ReLU) -> permeability."""

    def __init__(self, input_dim, hidden_dim, output_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        """x: (batch_size, input_dim) -> (batch_size, output_dim)."""
        return self.fc2(self.relu(self.fc1(x)))

In [ ]:
def make_model(X, y, batch_size, hidden_size, learning_rate):
    """Split the data 80/20 and build the data loaders, model, loss and optimiser for one run."""
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

    def to_dataset(features, labels):
        return TensorDataset(torch.tensor(features, dtype=torch.float32),
                             torch.tensor(labels, dtype=torch.float32).unsqueeze(1))

    train_loader = DataLoader(to_dataset(X_train, y_train), batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(to_dataset(X_test, y_test), batch_size=batch_size, shuffle=False)

    model = PermeabilityMLP(X.shape[1], hidden_size, output_dim=1)
    criterion = nn.MSELoss()
    optimiser = optim.Adam(model.parameters(), lr=learning_rate)
    return model, train_loader, test_loader, criterion, optimiser

## Training and evaluation

In [ ]:
def train(model, train_loader, criterion, optimiser, num_epochs):
    """Train for `num_epochs`, logging the mean training loss of each epoch to W&B."""
    model.train()
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimiser.zero_grad()
            loss = criterion(model(batch_X), batch_y)
            loss.backward()
            optimiser.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        wandb.log({"loss": avg_loss})
        if epoch % 10 == 0:
            print(f"Epoch {epoch + 1}/{num_epochs} - Loss: {avg_loss:.4f}")

In [ ]:
def evaluate(model, test_loader):
    """Compute test-set MAE, MSE and R² and log them to W&B (the sweeps minimise "MSE")."""
    model.eval()
    predictions, actuals = [], []
    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            predictions.append(model(batch_X).numpy())
            actuals.append(batch_y.numpy())
    predictions = np.concatenate(predictions)
    actuals = np.concatenate(actuals)

    mae = mean_absolute_error(actuals, predictions)
    mse = mean_squared_error(actuals, predictions)
    r2 = r2_score(actuals, predictions)
    print(f"MAE: {mae:.4f}, MSE: {mse:.4f}, R2: {r2:.4f}")
    wandb.log({"MAE": mae, "MSE": mse, "R2": r2})

## Run the sweep

`model_pipeline` trains and evaluates one model per sweep trial. The cell after it re-runs the best configuration; pass `SWEEP_SEARCH` instead to repeat the search. A W&B account is required (`wandb login`).

In [ ]:
def model_pipeline():
    """W&B sweep entry point: train and evaluate one MLP with the run's hyperparameters."""
    with wandb.init(project=PROJECT_NAME) as run:
        config = run.config
        torch.manual_seed(SEED)
        np.random.seed(SEED)

        model, train_loader, test_loader, criterion, optimiser = make_model(
            X, y, config.batch_size, config.hidden_size, config.learning_rate
        )
        print("\nTraining...")
        train(model, train_loader, criterion, optimiser, config.epochs)
        print("\nEvaluating...")
        evaluate(model, test_loader)

In [51]:
wandb.login()
sweep_id = wandb.sweep(sweep=SWEEP_BEST, project=PROJECT_NAME)
wandb.agent(sweep_id, function=model_pipeline)


Training...
Epoch 1/50 - Loss: 7.7884
Epoch 11/50 - Loss: 0.9078
Epoch 21/50 - Loss: 0.8521
Epoch 31/50 - Loss: 0.8293
Epoch 41/50 - Loss: 0.8155

Evaluating...
MAE: 0.6074, MSE: 0.7705, R2: 0.3131


### Result

The best configuration reaches **R² = 0.313** (MAE 0.607, MSE 0.771) on the 20% hold-out split. The sweep chose the narrowest hidden layer it was offered (4 units, from options of 4–128), and accuracy matches the tree ensembles on the same fingerprints. This suggests the representation, not model capacity, limits performance.